In [1]:


import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from lazypredict.Supervised import LazyClassifier
from IPython.display import clear_output
# numpy and pandas for data manipulation

# File system manangement
import os

# Suppress warnings 
import warnings
warnings.filterwarnings('ignore')
from autogluon.tabular import TabularPredictor

# matplotlib and seaborn for plotting
import seaborn as sns
import plotly.express as px


# ------------------- IMPORT SRC ------------------------------------
# src is the parent folder of notebooks, so we need to add it to sys.path to import config and utils
import sys
notebook_dir = os.getcwd() 

# Parent folder of src
project_root = os.path.abspath(os.path.join(notebook_dir, "..")) 
sys.path.append(project_root)

print("sys.path contains:", sys.path[-1])

from src import config, utils  
# -------------------------------------------------------

X_train = pd.read_csv('../data/X_train.csv')
X_test = pd.read_csv('../data/X_test.csv')
y_train = pd.read_csv('../data/y_train.csv')

# -----------------------------
# 3️⃣ Combine X and y for AutoGluon
# AutoGluon expects a single DataFrame
# -----------------------------
train_data = X_train.copy()
train_data['target'] = y_train.squeeze()  # ensure Series, not DataFrame


cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)


y_train_int = y_train_enc

sys.path contains: /home/ismail/Documents/projects/ml-projects/binary-classification-with-a-bank-churn-dataset


<mark>2️⃣ Text features (raw text like reviews, comments)

If you have long free-form text (e.g., product reviews, tweets), AutoGluon can handle it too, but you need to tell it explicitly you have a text column.

It will automatically create text models (like bag-of-words, embeddings) and ensemble them with other models.

</mark>

If you want, I can make a ready-to-run AutoGluon template for your binary string target + categorical features dataset?

In [ ]:
#-----------------
# 5️⃣ Train AutoGluon TabularPredictor
# -----------------------------


# feature_metadata = {
#     'categorical': cat_cols,
#     'text': text_cols if 'text_cols' in locals() else []
# }

# predictor = TabularPredictor(label='target').fit(
#     train_data=train_data,
#     feature_metadata=feature_metadata
# )


predictor = TabularPredictor(
    label='target',
    problem_type='binary',
    eval_metric='roc_auc' # accuracy
).fit(
    train_data=train_data,
    # feature_metadata=feature_metadata,

    time_limit=600,       # 10 minutes
    presets='best_quality'  # keeps stacking + multiple models
    # num_cpus = 8
    #            num_stack_levels = 2, 
    # hyperparameters omitted → default used
    #     num_bag_folds=5,  # enables k-fold bagging for robust CV
    # stack_ensemble_levels=2  # create a two-level stacking ensemble
)
# -----------------------------
# 6️⃣ Check training summary
# -----------------------------
predictor.fit_summary()

# -----------------------------
# 7️⃣ Make predictions on X_test
# -----------------------------
predictions = predictor.predict(X_test)

# -----------------------------
# 8️⃣ Save predictions
# -----------------------------
# predictions.to_csv('submission.csv', index=False)

# -----------------------------
# 9️⃣ Optional: evaluate CV performance
# -----------------------------
cv_results = predictor.leaderboard(silent=True)
print(cv_results)


No path specified. Models will be saved in: "AutogluonModels/ag-20260309_000241"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.10.18
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Sun, 12 Oct 2025 12:45:18 +0000
CPU Count:          16
Pytorch Version:    2.8.0+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 3.68/3.68 GB
Total GPU Memory:   Free: 3.68 GB, Allocated: 0.00 GB, Total: 3.68 GB
GPU Count:          1
Memory Avail:       3.92 GB / 15.24 GB (25.7%)
Disk Space Avail:   3.47 GB / 215.22 GB (1.6%)
	We recommend a minimum available disk space of 10 GB, and large datasets may require more.
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_

(raylet) A worker died or was killed while executing a task by an unexpected system error. To troubleshoot the problem, check the logs for the dead worker. Lease ID: 4800000001000000ffffffffffffffffffffffffffffffffffffffffffffffff Worker ID: 92c827c44117da94fbaadbd0a12628bbdcc863bffee95f5322ceed94 Node ID: 76e8e6572e55f94b316ec94d55ada25dd46a29ba2141a016f09b03d9 Worker IP address: 192.168.1.8 Worker port: 39433 Worker PID: 2103997 Worker exit type: SYSTEM_ERROR Worker exit detail: Worker unexpectedly exits with a connection error code 2. End of file. Some common causes include: (1) the process was killed by the OOM killer due to high memory usage, (2) ray stop --force was called, or (3) the worker crashed unexpectedly due to SIGSEGV or another unexpected error.
(raylet) Task _ray_fit failed. There are 2 retries remaining, so the task will be retried. Error: 


	Time limit exceeded... Skipping NeuralNetFastAI_BAG_L1.
Fitting model: WeightedEnsemble_L2 ... Training model for up to 360.00s of the 110.61s of remaining time.
	Fitting 1 model on all data | Fitting with cpus=16, gpus=0, mem=0.0/4.4 GB
	Ensemble Weights: {'LightGBM_BAG_L1': 0.522, 'CatBoost_BAG_L1': 0.391, 'RandomForestGini_BAG_L1': 0.043, 'RandomForestEntr_BAG_L1': 0.043}
	0.894	 = Validation score   (roc_auc)
	3.06s	 = Training   runtime
	0.02s	 = Validation runtime
Fitting 108 L2 models, fit_strategy="sequential" ...
Fitting model: LightGBMXT_BAG_L2 ... Training model for up to 107.49s of the 107.43s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (8 workers, per: cpus=2, gpus=0, memory=3.27%)
	0.8935	 = Validation score   (roc_auc)
	10.3s	 = Training   runtime
	1.29s	 = Validation runtime
Fitting model: LightGBM_BAG_L2 ... Training model for up to 94.90s of the 94.85s of remaining time.
	Fitting 8 child models (S1F1 - S1F8

*** Summary of fit() ***
Estimated performance of each model:
                      model  score_val eval_metric  pred_time_val  fit_time  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0       WeightedEnsemble_L3       0.89     roc_auc          30.47    306.92                    0.02               5.33            3       True         14
1       WeightedEnsemble_L2       0.89     roc_auc          11.00    212.88                    0.02               3.06            2       True          8
2           LightGBM_BAG_L2       0.89     roc_auc          21.32    255.89                    0.50               6.61            2       True         10
3         LightGBMXT_BAG_L2       0.89     roc_auc          22.11    259.59                    1.29              10.30            2       True          9
4           LightGBM_BAG_L1       0.89     roc_auc           3.14     19.44                    3.14              19.44            1       True          2
5           Ca

(raylet) The node with node id: 76e8e6572e55f94b316ec94d55ada25dd46a29ba2141a016f09b03d9 and address: 192.168.1.8 and node name: 192.168.1.8 has been marked dead because the detector has missed too many heartbeats from it. This can happen when a 	(1) raylet crashes unexpectedly (OOM, etc.) 
	(2) raylet has lagging heartbeats due to slow network or busy workload.
(raylet) Raylet is terminated. Termination is unexpected. Possible reasons include: (1) SIGKILL by the user or system OOM killer, (2) Invalid memory access from Raylet causing SIGSEGV or SIGBUS, (3) Other termination signals. Last 20 lines of the Raylet logs:
    *** StackTrace Information ***
    /home/ismail/miniconda/envs/ml-dl/lib/python3.10/site-packages/ray/core/src/ray/raylet/raylet(+0xf4569a) [0x557fdad0a69a] ray::operator<<()
    /home/ismail/miniconda/envs/ml-dl/lib/python3.10/site-packages/ray/core/src/ray/raylet/raylet(+0xf47cb9) [0x557fdad0ccb9] ray::RayLog::~RayLog()
    /home/ismail/miniconda/envs/ml-dl/lib/pytho

In [6]:
pd.set_option('display.float_format', lambda x: f'{x:.5f}')
print(cv_results)

                      model  score_val eval_metric  pred_time_val  fit_time  \
0       WeightedEnsemble_L3    0.89466     roc_auc       30.46668 306.91689   
1       WeightedEnsemble_L2    0.89400     roc_auc       10.99636 212.88074   
2           LightGBM_BAG_L2    0.89386     roc_auc       21.32197 255.89448   
3         LightGBMXT_BAG_L2    0.89349     roc_auc       22.10833 259.58725   
4           LightGBM_BAG_L1    0.89341     roc_auc        3.14432  19.44231   
5           CatBoost_BAG_L2    0.89339     roc_auc       20.89375 272.56270   
6           CatBoost_BAG_L1    0.89317     roc_auc        0.06628 159.59970   
7         LightGBMXT_BAG_L1    0.89147     roc_auc        4.38115  24.76242   
8   RandomForestEntr_BAG_L2    0.89101     roc_auc       25.27261 272.04202   
9   RandomForestGini_BAG_L2    0.89030     roc_auc       25.49758 272.21767   
10    ExtraTreesEntr_BAG_L1    0.88434     roc_auc        3.50567   9.19667   
11  RandomForestGini_BAG_L1    0.88380     roc_auc  

In [ ]:
prediction = predictor.predict_proba(X_test)

data_submit = pd.read_csv('../sample_submission.csv')
data_submit.Exited = prediction[1]
data_submit[['id', 'Exited']].to_csv('../outputs/submissions/autoglean.csv', index=False)


Perfect — your code already works, but we can make it **“advanced”** in several ways to **maximize performance, get more insights, and make it production-ready**. Here’s how:

---

## **1️⃣ Explicit Hyperparameter Tuning**

Instead of using default models, you can define **which models to train** and their **hyperparameters**, including boosting parameters, neural nets, and stacking options:

```python
hyperparameters = {
    'GBM': {'num_boost_round': 500, 'learning_rate': 0.05, 'num_leaves': 31},
    'CAT': {'iterations': 500, 'learning_rate': 0.05, 'depth': 6},
    'XGB': {'n_estimators': 500, 'learning_rate': 0.05, 'max_depth': 6},
    'RF': {'n_estimators': 200, 'max_depth': None},
    'NN_TORCH': {'epochs': 20, 'learning_rate': 0.001},
}
```

Then pass it to `fit()`:

```python
predictor = TabularPredictor(label='target', eval_metric='accuracy').fit(
    train_data=train_data,
    time_limit=1200,  # increase time if dataset is small
    hyperparameters=hyperparameters,
    presets='best_quality',
    num_bag_folds=5,  # enables k-fold bagging for robust CV
    stack_ensemble_levels=2  # create a two-level stacking ensemble
)
```

---

## **2️⃣ Enable K-Fold Bagging & Stacking**

* **Bagging (`num_bag_folds`)** improves stability on small datasets.
* **Stacked ensembles (`stack_ensemble_levels`)** improve overall accuracy by combining multiple models.

---

## **3️⃣ Handle Categorical/Text Explicitly**

If you have string/categorical features or text, AutoGluon usually detects them automatically, but you can be explicit:

```python
feature_metadata = {
    'categorical': cat_cols,
    'text': text_cols if 'text_cols' in locals() else []
}

predictor = TabularPredictor(label='target').fit(
    train_data=train_data,
    feature_metadata=feature_metadata
)
```

---

## **4️⃣ Save & Reload Models**

* Save the predictor to reuse later without retraining:

```python
predictor.save('autogluon_model')
# reload later
# predictor = TabularPredictor.load('autogluon_model')
```

---

## **5️⃣ Get Out-of-Fold (OOF) Predictions**

This is useful if you want to **stack your own models or analyze CV performance**:

```python
oof_preds = predictor.get_oof_pred_proba()
```

---

## **6️⃣ Evaluate & Compare Models**

* `leaderboard()` shows **all trained models, validation scores, hyperparameters**:

```python
cv_results = predictor.leaderboard(train_data, extra_metrics=['f1', 'roc_auc'])
print(cv_results)
```

* You can sort by any metric to see the **best performing model**.

---

## **7️⃣ Make Probabilistic Predictions**

Instead of just class labels, get probabilities for each class:

```python
proba_preds = predictor.predict_proba(X_test)
```

---

## **8️⃣ Advanced Prediction Example**

```python
predictions = predictor.predict(X_test)
proba_preds = predictor.predict_proba(X_test)

# save both
predictions.to_csv('submission.csv', index=False)
proba_preds.to_csv('submission_proba.csv', index=False)
```

---

✅ **Summary of Advanced Features**

| Feature                               | Why Use It                             |
| ------------------------------------- | -------------------------------------- |
| Hyperparameters dict                  | Customize models for better accuracy   |
| num_bag_folds & stack_ensemble_levels | More robust, better generalization     |
| feature_metadata                      | Explicit categorical/text handling     |
| OOF predictions                       | Useful for stacking or analysis        |
| predict_proba                         | Useful for probability-based scoring   |
| save/load model                       | Reuse trained models, avoid retraining |
| leaderboard with extra_metrics        | Compare models using multiple metrics  |

---

If you want, I can **rewrite your notebook fully “advanced”** with all these features applied **ready to run on your X_train/X_test/y_train dataset**.

Do you want me to do that?
